# TriaGS + GT Depth Supervision 파이프라인

기존 TriaGS에 **GT depth 활용**을 추가한 버전입니다.

### 기존 대비 개선점
1. **Depth back-projection으로 초기 포인트 생성** — 랜덤 → 표면 위 정확한 점
2. **Depth supervision loss** — 렌더링 depth vs GT depth 직접 비교
3. **Depth > 0 → 오브젝트 마스크** — 배경 자동 분리

---

| 단계 | 설명 |
|------|------|
| **0** | 환경 설정 |
| **1** | RGB + Depth 데이터 다운로드 & 변환 |
| **2** | 학습 (depth supervision 포함) |
| **3** | 메쉬 추출 |
| **4** | 렌더링 + 평가 |
| **5** | 시각화 |
| **6** | 결과 저장 |

---
## 0. 환경 설정

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("GPU 필요!")

In [ ]:
import os

WORK_DIR = "/content/triags"
DATA_DIR = "/content/data"
OUTPUT_DIR = "/content/output"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
if not os.path.exists(WORK_DIR):
    !git clone https://github.com/BAEJUNHAK/triags {WORK_DIR}
os.chdir(WORK_DIR)

In [ ]:
!pip install -q open3d trimesh mediapy scikit-image opencv-python plyfile tqdm tensorboard gdown
!pip install -q submodules/diff-gaussian-rasterization
!pip install -q git+https://github.com/camenduru/simple-knn

---
## 1. 데이터 다운로드 & 변환 (RGB + Depth)

In [ ]:
import gdown

# RGB + Camera
RGB_ZIP = f"{DATA_DIR}/dataset.zip"
if not os.path.exists(RGB_ZIP):
    gdown.download("https://drive.google.com/uc?id=1dnj1s-mqIuS6OcdSr5CczBr9u8yBzUUB", RGB_ZIP, quiet=False)

# Depth
DEPTH_ZIP = f"{DATA_DIR}/depth.zip"
if not os.path.exists(DEPTH_ZIP):
    gdown.download("https://drive.google.com/uc?id=1KmXCzBYv_mkPmZnWka1ThCZSNHTyivKQ", DEPTH_ZIP, quiet=False)

# 압축 해제
!unzip -q -o {RGB_ZIP} -d {DATA_DIR}/raw_rgb
!unzip -q -o {DEPTH_ZIP} -d {DATA_DIR}/raw_depth
print("다운로드 완료")

In [ ]:
import glob

def find_data_root(base_dir, prefix):
    for root, dirs, files in os.walk(base_dir):
        if any(f.startswith(prefix) for f in files):
            return root
    return base_dir

RGB_DATA_DIR = find_data_root(f"{DATA_DIR}/raw_rgb", "calib_")
DEPTH_DATA_DIR = find_data_root(f"{DATA_DIR}/raw_depth", "depth_raw_")

print(f"RGB 폴더:   {RGB_DATA_DIR}")
print(f"Depth 폴더: {DEPTH_DATA_DIR}")
print(f"RGB 파일:   {len(glob.glob(f'{RGB_DATA_DIR}/rgb_*.png'))}개")
print(f"Depth 파일: {len(glob.glob(f'{DEPTH_DATA_DIR}/depth_raw_*.png'))}개")
print(f"Calib 파일: {len(glob.glob(f'{RGB_DATA_DIR}/calib_*.ini'))}개")

In [ ]:
#@title 데이터 설정 {display-mode: "form"}

NUM_TRAIN = 160  #@param {type:"integer"}
NUM_TEST = 40  #@param {type:"integer"}
DEPTH_SCALE = 0.01  #@param {type:"number"}
#@markdown ↑ uint16 raw값 × DEPTH_SCALE = 실제 거리

NUM_TOTAL = NUM_TRAIN + NUM_TEST
print(f"Train: {NUM_TRAIN}, Test: {NUM_TEST}, 총: {NUM_TOTAL}")

In [ ]:
import numpy as np
import cv2
from pathlib import Path
import shutil

SCENE_DIR = f"{DATA_DIR}/scene"

def parse_calib(filepath):
    config = {}
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('[') or line.startswith('#'):
                continue
            key, value = line.split('=', 1)
            config[key] = value
    k_vals = list(map(float, config['k_matrix'].split(':')[-1].split(',')))
    K = np.array(k_vals).reshape(3, 3)
    r_vals = list(map(float, config['r_matrix'].split(':')[-1].split(',')))
    R = np.array(r_vals).reshape(3, 3)
    t_vals = list(map(float, config['t_vector'].split(':')[-1].split(',')))
    T = np.array(t_vals)
    return K, R, T, int(config['width']), int(config['height'])

def rotmat2qvec(R):
    Rxx, Ryx, Rzx, Rxy, Ryy, Rzy, Rxz, Ryz, Rzz = R.flat
    K = np.array([
        [Rxx - Ryy - Rzz, 0, 0, 0],
        [Ryx + Rxy, Ryy - Rxx - Rzz, 0, 0],
        [Rzx + Rxz, Rzy + Ryz, Rzz - Rxx - Ryy, 0],
        [Ryz - Rzy, Rzx - Rxz, Rxy - Ryx, Rxx + Ryy + Rzz]]) / 3.0
    eigvals, eigvecs = np.linalg.eigh(K)
    qvec = eigvecs[[3, 0, 1, 2], np.argmax(eigvals)]
    if qvec[0] < 0:
        qvec *= -1
    return qvec

# 파일 매칭
def get_index(filepath):
    return Path(filepath).stem.split('_')[-1]

calib_dict = {get_index(f): f for f in sorted(glob.glob(f"{RGB_DATA_DIR}/calib_*.ini"))}
rgb_dict = {get_index(f): f for f in sorted(glob.glob(f"{RGB_DATA_DIR}/rgb_*.png"))}
depth_dict = {get_index(f): f for f in sorted(glob.glob(f"{DEPTH_DATA_DIR}/depth_raw_*.png"))}

# 세 파일 모두 있는 인덱스만
all_indices = sorted(set(calib_dict.keys()) & set(rgb_dict.keys()) & set(depth_dict.keys()))
N_ALL = len(all_indices)
print(f"RGB+Depth+Calib 쌍: {N_ALL}개")

# 균일 샘플링 + 인터리빙 분할
if NUM_TOTAL < N_ALL:
    sample_idx = np.linspace(0, N_ALL - 1, NUM_TOTAL, dtype=int)
    selected_indices = [all_indices[i] for i in sample_idx]
else:
    selected_indices = all_indices

stride = max(1, len(selected_indices) // NUM_TEST)
train_indices = []
test_indices = []
for i, idx in enumerate(selected_indices):
    if (i + 1) % stride == 0:
        test_indices.append(idx)
    else:
        train_indices.append(idx)

print(f"Train: {len(train_indices)}, Test: {len(test_indices)}")

# === 적응적 분할: 고립된 test 뷰를 train으로 swap ===
SWAP_THRESHOLD = 60  # nearest train 거리 기준

def get_cam_pos(idx):
    _, R, T, _, _ = parse_calib(calib_dict[idx])
    return -R.T @ T

train_positions = np.array([get_cam_pos(t) for t in train_indices])
n_swapped = 0

for i in range(len(test_indices)):
    test_pos = get_cam_pos(test_indices[i])
    dists = np.linalg.norm(train_positions - test_pos, axis=1)
    min_dist = dists.min()

    if min_dist > SWAP_THRESHOLD:
        nearest_train_i = dists.argmin()
        print(f"  Swap: test {test_indices[i]} (dist={min_dist:.1f}) <-> train {train_indices[nearest_train_i]}")
        test_indices[i], train_indices[nearest_train_i] = train_indices[nearest_train_i], test_indices[i]
        # train_positions 업데이트
        train_positions[nearest_train_i] = get_cam_pos(train_indices[nearest_train_i])
        n_swapped += 1

if n_swapped > 0:
    print(f"적응적 분할: {n_swapped}개 뷰 swap")
else:
    print("적응적 분할: swap 필요 없음")
print(f"최종 Train: {len(train_indices)}, Test: {len(test_indices)}")

In [ ]:
# ============================================================
# COLMAP 변환 + depth back-projection 초기 포인트 생성
# ============================================================
if os.path.exists(SCENE_DIR):
    shutil.rmtree(SCENE_DIR)

images_dir = os.path.join(SCENE_DIR, "images")
sparse_dir = os.path.join(SCENE_DIR, "sparse")
depth_out_dir = os.path.join(SCENE_DIR, "depth")  # TriaGS가 읽을 depth 폴더
test_images_dir = os.path.join(SCENE_DIR, "test_images")
os.makedirs(images_dir, exist_ok=True)
os.makedirs(sparse_dir, exist_ok=True)
os.makedirs(depth_out_dir, exist_ok=True)
os.makedirs(test_images_dir, exist_ok=True)

np.random.seed(42)

# cameras.txt
K0, _, _, W0, H0 = parse_calib(calib_dict[train_indices[0]])
fx, fy, cx, cy = K0[0, 0], K0[1, 1], K0[0, 2], K0[1, 2]

with open(os.path.join(sparse_dir, "cameras.txt"), 'w') as f:
    f.write("# Camera list with one line of data per camera:\n")
    f.write("# CAMERA_ID, MODEL, WIDTH, HEIGHT, PARAMS[]\n")
    f.write(f"# Number of cameras: 1\n")
    f.write(f"1 PINHOLE {W0} {H0} {fx} {fy} {cx} {cy}\n")

# images.txt + depth 복사 + 이미지 (배경 제거)
camera_positions = []
all_backproj_points = []
all_backproj_colors = []

# depth back-projection용 샘플 수 (전체 하면 너무 많으므로 뷰당 제한)
POINTS_PER_VIEW = 2000

with open(os.path.join(sparse_dir, "images.txt"), 'w') as f:
    f.write("# Image list with two lines of data per image:\n")
    f.write("# IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME\n")
    f.write("# POINTS2D[] as (X, Y, POINT3D_ID)\n")
    f.write(f"# Number of images: {len(train_indices)}\n")

    for img_id, idx in enumerate(train_indices, start=1):
        K, R, T, W, H = parse_calib(calib_dict[idx])
        qvec = rotmat2qvec(R)
        cam_pos = -R.T @ T
        camera_positions.append(cam_pos)

        img_name = f"rgb_{idx}.png"

        # --- 배경 제거: depth > 0 마스크로 RGB 배경을 검정으로 교체 ---
        rgb_img = cv2.imread(rgb_dict[idx])
        depth_src = depth_dict.get(idx)
        if depth_src:
            depth_raw = cv2.imread(depth_src, cv2.IMREAD_UNCHANGED)
            bg_mask = (depth_raw > 0).astype(np.float32)[..., np.newaxis]  # (H, W, 1)
            rgb_masked = (rgb_img * bg_mask).astype(np.uint8)  # 배경 → [0,0,0]
            cv2.imwrite(os.path.join(images_dir, img_name), rgb_masked)
        else:
            # depth 없으면 원본 복사
            cv2.imwrite(os.path.join(images_dir, img_name), rgb_img)

        # depth 파일 심볼릭 링크
        if depth_src:
            depth_dst_name = f"depth_raw_rgb_{idx}.png"
            os.symlink(os.path.abspath(depth_src), os.path.join(depth_out_dir, depth_dst_name))

        f.write(f"{img_id} {qvec[0]} {qvec[1]} {qvec[2]} {qvec[3]} {T[0]} {T[1]} {T[2]} 1 {img_name}\n")
        f.write("\n")

        # ---- Depth back-projection: 표면 위 3D 점 생성 ----
        if depth_src:
            depth_m = depth_raw.astype(np.float32) * DEPTH_SCALE

            # 유효 픽셀 (depth > 0)
            valid_mask = depth_m > 0
            ys, xs = np.where(valid_mask)

            if len(xs) > POINTS_PER_VIEW:
                choice = np.random.choice(len(xs), POINTS_PER_VIEW, replace=False)
                xs, ys = xs[choice], ys[choice]

            depths = depth_m[ys, xs]

            # 카메라 좌표계로 역투영: X_cam = K^-1 @ [u, v, 1] * depth
            pts_cam = np.stack([
                (xs - cx) / fx * depths,
                (ys - cy) / fy * depths,
                depths
            ], axis=1)  # (N, 3)

            # 월드 좌표로 변환: X_world = R^T @ (X_cam - T)
            pts_world = (R.T @ (pts_cam.T - T[:, None])).T  # (N, 3)

            all_backproj_points.append(pts_world)

            # RGB 색상 가져오기 (마스킹된 이미지에서)
            rgb_for_color = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2RGB)
            colors = rgb_for_color[ys, xs]  # (N, 3) uint8
            all_backproj_colors.append(colors)

# test 이미지도 배경 제거 후 저장
for idx in test_indices:
    rgb_img = cv2.imread(rgb_dict[idx])
    depth_src = depth_dict.get(idx)
    if depth_src:
        depth_raw = cv2.imread(depth_src, cv2.IMREAD_UNCHANGED)
        bg_mask = (depth_raw > 0).astype(np.float32)[..., np.newaxis]
        rgb_masked = (rgb_img * bg_mask).astype(np.uint8)
        cv2.imwrite(os.path.join(test_images_dir, f"rgb_{idx}.png"), rgb_masked)
    else:
        cv2.imwrite(os.path.join(test_images_dir, f"rgb_{idx}.png"), rgb_img)

camera_positions = np.array(camera_positions)
print(f"[OK] cameras.txt, images.txt (Train {len(train_indices)}장)")
print(f"[OK] depth/ ({len(os.listdir(depth_out_dir))}개 depth 파일)")
print(f"[OK] test_images/ (Test {len(test_indices)}장)")
print(f"[OK] 배경 제거 완료: depth > 0 마스크 적용 (배경 → 검정)")

In [ ]:
# ============================================================
# points3D.ply: depth back-projection 기반 초기 포인트
# ============================================================
from plyfile import PlyData, PlyElement

np.random.seed(42)

all_pts = np.concatenate(all_backproj_points, axis=0).astype(np.float32)
all_rgb = np.concatenate(all_backproj_colors, axis=0).astype(np.uint8)

print(f"Back-projected 포인트: {len(all_pts):,}개")

# 너무 많으면 서브샘플링 (GPU 메모리 고려)
MAX_INIT_POINTS = 200000
if len(all_pts) > MAX_INIT_POINTS:
    choice = np.random.choice(len(all_pts), MAX_INIT_POINTS, replace=False)
    all_pts = all_pts[choice]
    all_rgb = all_rgb[choice]
    print(f"서브샘플링 → {len(all_pts):,}개")

# 이상치 제거 (중심에서 너무 먼 점)
center = np.median(all_pts, axis=0)
dists = np.linalg.norm(all_pts - center, axis=1)
threshold = np.percentile(dists, 99)
valid = dists < threshold
all_pts = all_pts[valid]
all_rgb = all_rgb[valid]
print(f"이상치 제거 후: {len(all_pts):,}개")
print(f"포인트 범위: {all_pts.min(axis=0)} ~ {all_pts.max(axis=0)}")

# PLY 저장
with open(os.path.join(sparse_dir, "points3D.txt"), 'w') as f:
    f.write("# 3D point list\n")

ply_path = os.path.join(sparse_dir, "points3D.ply")
dtype = [('x','f4'),('y','f4'),('z','f4'),('nx','f4'),('ny','f4'),('nz','f4'),('red','u1'),('green','u1'),('blue','u1')]
normals = np.zeros_like(all_pts, dtype=np.float32)
elements = np.empty(len(all_pts), dtype=dtype)
elements[:] = list(map(tuple, np.concatenate([all_pts, normals, all_rgb], axis=1)))
PlyData([PlyElement.describe(elements, 'vertex')]).write(ply_path)

print(f"\n[OK] points3D.ply ({len(all_pts):,} points, depth back-projection)")
print(f"     vs 이전: 랜덤 100,000점")

In [ ]:
# 초기 포인트 시각화
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(12, 5))

# 3D 포인트 클라우드
ax1 = fig.add_subplot(121, projection='3d')
step = max(1, len(all_pts) // 5000)
ax1.scatter(all_pts[::step, 0], all_pts[::step, 1], all_pts[::step, 2],
            c=all_rgb[::step] / 255.0, s=1, alpha=0.5)
ax1.set_title(f'초기 포인트 ({len(all_pts):,}개, depth back-proj)')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

# 카메라 위치
ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(camera_positions[::2, 0], camera_positions[::2, 1], camera_positions[::2, 2],
            c='blue', s=5, alpha=0.5, label='Cameras')
ax2.scatter(all_pts[::step*5, 0], all_pts[::step*5, 1], all_pts[::step*5, 2],
            c='gray', s=0.5, alpha=0.2, label='Points')
ax2.legend()
ax2.set_title('카메라 + 포인트 배치')

plt.tight_layout()
plt.show()

---
## 2. 학습 (Depth Supervision 포함)

In [ ]:
#@title 학습 파라미터 {display-mode: "form"}

ITERATIONS = 30000  #@param {type:"integer"}
RESOLUTION = 2  #@param [1, 2, 4] {type:"raw"}
USE_APPEARANCE = True  #@param {type:"boolean"}

TGPC_WEIGHT = 0.1  #@param {type:"number"}
TGPC_NEIGHBORS = 12  #@param {type:"integer"}
MULTI_VIEW_NUM = 8  #@param {type:"integer"}
MULTI_VIEW_MAX_ANGLE = 30  #@param {type:"integer"}
MULTI_VIEW_MIN_DIS = 1.0  #@param {type:"number"}
MULTI_VIEW_MAX_DIS = 50.0  #@param {type:"number"}

LAMBDA_DSSIM = 0.2  #@param {type:"number"}
LAMBDA_DEPTH_NORMAL = 0.05  #@param {type:"number"}
LAMBDA_GEO = 0.03  #@param {type:"number"}
LAMBDA_NCC = 0.15  #@param {type:"number"}

#@markdown ### Depth Supervision (NEW)
LAMBDA_GT_DEPTH = 0.002  #@param {type:"number"}

SOURCE_PATH = SCENE_DIR
MODEL_PATH = f"{OUTPUT_DIR}/custom_scene_depth"

print(f"Source: {SOURCE_PATH}")
print(f"Output: {MODEL_PATH}")
print(f"Depth supervision: lambda={LAMBDA_GT_DEPTH}")

In [ ]:
os.chdir(WORK_DIR)

cmd = f"""python train.py \
    -s {SOURCE_PATH} \
    -m {MODEL_PATH} \
    -r {RESOLUTION} \
    --iterations {ITERATIONS} \
    --use_gt_depth \
    --depth_scale {DEPTH_SCALE} \
    --lambda_gt_depth {LAMBDA_GT_DEPTH} \
    --lambda_dssim {LAMBDA_DSSIM} \
    --lambda_depth_normal {LAMBDA_DEPTH_NORMAL} \
    --multi_view_geo_weight {LAMBDA_GEO} \
    --multi_view_ncc_weight {LAMBDA_NCC} \
    --tgpc_loss_weight {TGPC_WEIGHT} \
    --tgpc_num_neighbors {TGPC_NEIGHBORS} \
    --multi_view_num {MULTI_VIEW_NUM} \
    --multi_view_max_angle {MULTI_VIEW_MAX_ANGLE} \
    --multi_view_min_dis {MULTI_VIEW_MIN_DIS} \
    --multi_view_max_dis {MULTI_VIEW_MAX_DIS} \
    --test_iterations 7000 15000 30000 \
    --save_iterations 15000 25000 30000 \
    --checkpoint_iterations 15000"""

if USE_APPEARANCE:
    cmd += " --use_decoupled_appearance"

print(cmd)

In [ ]:
!{cmd}

---
## 3. 메쉬 추출

In [ ]:
os.chdir(WORK_DIR)
!python mesh_extract.py -s {SOURCE_PATH} -m {MODEL_PATH}

In [ ]:
import trimesh
for mf in glob.glob(f"{MODEL_PATH}/**/fuse_*_post.ply", recursive=True) + \
          glob.glob(f"{MODEL_PATH}/**/recon.ply", recursive=True):
    mesh = trimesh.load(mf)
    print(f"{mf}")
    print(f"  Vertices: {len(mesh.vertices):,}, Faces: {len(mesh.faces):,}")

---
## 4. 렌더링 + 평가

In [ ]:
import torch
import torchvision
import json
import numpy as np
from PIL import Image
import torchvision.transforms.functional as tf
from tqdm import tqdm

os.chdir(WORK_DIR)

from scene import GaussianModel
from gaussian_renderer import render as gs_render
from arguments import ModelParams, PipelineParams
from argparse import ArgumentParser
from scene.cameras import Camera as GSCamera
from utils.graphics_utils import focal2fov
from utils.loss_utils import ssim
from utils.image_utils import psnr
from lpipsPyTorch import lpips

parser = ArgumentParser()
lp = ModelParams(parser)
pp = PipelineParams(parser)
args = parser.parse_args(["-s", SOURCE_PATH, "-m", MODEL_PATH])
dataset = lp.extract(args)
pipe = pp.extract(args)

bg_color = torch.tensor([0, 0, 0], dtype=torch.float32, device="cuda")
kernel_size = dataset.kernel_size

# 평가할 iteration 목록 (저장된 것만)
pc_dir = os.path.join(MODEL_PATH, "point_cloud")
saved_iters = sorted([int(d.split("_")[-1]) for d in os.listdir(pc_dir) if d.startswith("iteration_")])
print(f"저장된 iterations: {saved_iters}")

for eval_iter in saved_iters:
    print(f"\n{'='*50}")
    print(f"Evaluating iteration {eval_iter}")
    print(f"{'='*50}")

    gaussians = GaussianModel(dataset.sh_degree)
    gaussians.load_ply(os.path.join(pc_dir, f"iteration_{eval_iter}", "point_cloud.ply"))
    print(f"모델 로드: {gaussians.get_xyz.shape[0]:,} Gaussians")

    test_render_dir = os.path.join(MODEL_PATH, "test", f"ours_{eval_iter}", "renders")
    test_gt_dir = os.path.join(MODEL_PATH, "test", f"ours_{eval_iter}", "gt")
    os.makedirs(test_render_dir, exist_ok=True)
    os.makedirs(test_gt_dir, exist_ok=True)

    psnrs, ssims, lpipss = [], [], []

    for ti, idx in enumerate(tqdm(test_indices, desc=f"Test rendering (iter {eval_iter})")):
        K, R, T, W, H = parse_calib(calib_dict[idx])
        scale = RESOLUTION
        W_s, H_s = W // scale, H // scale
        fx_s, fy_s = K[0,0] / scale, K[1,1] / scale
        FovX = focal2fov(fx_s, W_s)
        FovY = focal2fov(fy_s, H_s)

        # GT 이미지: 배경 제거된 test 이미지 사용 (검정 배경, 렌더링과 일치)
        test_img_path = os.path.join(SCENE_DIR, "test_images", f"rgb_{idx}.png")
        if os.path.exists(test_img_path):
            gt_img = Image.open(test_img_path).resize((W_s, H_s), Image.LANCZOS)
        else:
            # fallback: 원본에 depth 마스크 적용
            gt_img = Image.open(rgb_dict[idx]).resize((W_s, H_s), Image.LANCZOS)
            depth_src = depth_dict.get(idx)
            if depth_src:
                import cv2
                depth_raw = cv2.imread(depth_src, cv2.IMREAD_UNCHANGED)
                depth_resized = cv2.resize(depth_raw, (W_s, H_s), interpolation=cv2.INTER_NEAREST)
                mask = (depth_resized > 0).astype(np.float32)
                gt_arr = np.array(gt_img).astype(np.float32) * mask[..., np.newaxis]
                gt_img = Image.fromarray(gt_arr.astype(np.uint8))

        gt_tensor = tf.to_tensor(gt_img).cuda()

        cam = GSCamera(
            colmap_id=ti, R=R.T, T=T,
            FoVx=FovX, FoVy=FovY,
            image=gt_tensor, gt_alpha_mask=None,
            image_name=f"test_{idx}", image_path=rgb_dict[idx], uid=ti,
            data_device="cuda"
        )

        with torch.no_grad():
            out = gs_render(cam, gaussians, pipe, bg_color, kernel_size=kernel_size)
            rendered = out["render"].clamp(0.0, 1.0)

        psnrs.append(psnr(rendered.unsqueeze(0), gt_tensor.unsqueeze(0)).item())
        ssims.append(ssim(rendered.unsqueeze(0), gt_tensor.unsqueeze(0)).item())
        lpipss.append(lpips(rendered.unsqueeze(0), gt_tensor.unsqueeze(0), net_type="vgg").item())

        torchvision.utils.save_image(rendered, os.path.join(test_render_dir, f"test_{idx}.png"))
        torchvision.utils.save_image(gt_tensor, os.path.join(test_gt_dir, f"test_{idx}.png"))

    print(f"\n  PSNR:  {np.mean(psnrs):.3f} dB")
    print(f"  SSIM:  {np.mean(ssims):.4f}")
    print(f"  LPIPS: {np.mean(lpipss):.4f}")

    test_results = {
        "test_metrics": {
            "PSNR": float(np.mean(psnrs)),
            "SSIM": float(np.mean(ssims)),
            "LPIPS": float(np.mean(lpipss)),
            "num_test_views": len(test_indices),
            "num_train_views": len(train_indices),
            "iteration": eval_iter,
            "num_gaussians": gaussians.get_xyz.shape[0],
        }
    }
    with open(os.path.join(MODEL_PATH, f"test_results_iter{eval_iter}.json"), "w") as f:
        json.dump(test_results, f, indent=2)
    print(f"  저장: test_results_iter{eval_iter}.json")

---
## 4.5 기하학적 정합도 평가

test 뷰에서 렌더링 depth vs GT depth를 비교하여 기하학적 정확도를 측정합니다.

| 지표 | 의미 | 좋은 값 |
|------|------|--------|
| AbsRel | 상대 오차 `\|pred-gt\|/gt` | 낮을수록 |
| SqRel | 제곱 상대 오차 `(pred-gt)²/gt` | 낮을수록 |
| RMSE | 절대 오차 | 낮을수록 |
| δ < 1.25 | 정확도 비율 | 높을수록 |
| Normal <30° | depth에서 계산한 법선 정확도 | 높을수록 |
| Scale std | 뷰 간 스케일 일관성 | 낮을수록 |

In [ ]:
# 기하학적 정합도 평가: Per-view 상세 분석 (모든 저장된 iteration)
import torch
import numpy as np
import cv2
from tqdm import tqdm

os.chdir(WORK_DIR)

def depth_to_normals(depth, fx, fy):
    dz_dy = np.gradient(depth, axis=0)
    dz_dx = np.gradient(depth, axis=1)
    normals = np.stack([-dz_dx / fx, -dz_dy / fy, np.ones_like(depth)], axis=-1)
    norm = np.linalg.norm(normals, axis=-1, keepdims=True).clip(1e-8)
    return normals / norm

pc_dir = os.path.join(MODEL_PATH, "point_cloud")
saved_iters = sorted([int(d.split("_")[-1]) for d in os.listdir(pc_dir) if d.startswith("iteration_")])
print(f"저장된 iterations: {saved_iters}")

for eval_iter in saved_iters:
    print(f"\n{'='*60}")
    print(f"기하학 평가: iteration {eval_iter}")
    print(f"{'='*60}")

    gaussians = GaussianModel(dataset.sh_degree)
    gaussians.load_ply(os.path.join(pc_dir, f"iteration_{eval_iter}", "point_cloud.ply"))
    print(f"모델 로드: {gaussians.get_xyz.shape[0]:,} Gaussians")

    # Pass 1: Global scale factor
    all_pred_medians = []
    all_gt_medians = []

    for ti, idx in enumerate(tqdm(test_indices, desc=f"Scale estimation (iter {eval_iter})")):
        K, R, T, W, H = parse_calib(calib_dict[idx])
        scale_r = RESOLUTION
        W_s, H_s = W // scale_r, H // scale_r
        fx_s, fy_s = K[0,0] / scale_r, K[1,1] / scale_r
        FovX = focal2fov(fx_s, W_s)
        FovY = focal2fov(fy_s, H_s)

        gt_img = Image.open(rgb_dict[idx]).resize((W_s, H_s), Image.LANCZOS)
        gt_tensor = tf.to_tensor(gt_img).cuda()

        cam = GSCamera(
            colmap_id=ti, R=R.T, T=T,
            FoVx=FovX, FoVy=FovY,
            image=gt_tensor, gt_alpha_mask=None,
            image_name=f"test_{idx}", image_path=rgb_dict[idx], uid=ti,
            data_device="cuda"
        )

        with torch.no_grad():
            out = gs_render(cam, gaussians, pipe, bg_color, kernel_size=kernel_size,
                           require_depth=True)
            rendered_depth = out["expected_depth"].squeeze(0).cpu().numpy()

        depth_src = depth_dict.get(idx)
        if depth_src is None:
            continue
        gt_depth = cv2.imread(depth_src, cv2.IMREAD_UNCHANGED).astype(np.float32) * DEPTH_SCALE
        gt_depth = cv2.resize(gt_depth, (W_s, H_s), interpolation=cv2.INTER_NEAREST)

        valid = gt_depth > 0
        if valid.sum() < 100:
            continue

        all_pred_medians.append(np.median(rendered_depth[valid]))
        all_gt_medians.append(np.median(gt_depth[valid]))

    global_scale = np.median(np.array(all_gt_medians) / np.array(all_pred_medians)) if all_pred_medians else 1.0
    print(f"Global scale factor: {global_scale:.4f}")

    # Pass 2: Per-view metrics
    per_view = {
        'idx': [], 'abs_rel': [], 'sq_rel': [], 'rmse': [],
        'delta_1': [], 'delta_2': [], 'delta_3': [],
        'scale_factor': [], 'normal_acc': [],
    }
    np.random.seed(42)
    all_gt_depths_sampled = []
    all_errors_sampled = []
    all_rel_errors_sampled = []

    for ti, idx in enumerate(tqdm(test_indices, desc=f"Geometry eval (iter {eval_iter})")):
        K, R, T, W, H = parse_calib(calib_dict[idx])
        scale_r = RESOLUTION
        W_s, H_s = W // scale_r, H // scale_r
        fx_s, fy_s = K[0,0] / scale_r, K[1,1] / scale_r
        FovX = focal2fov(fx_s, W_s)
        FovY = focal2fov(fy_s, H_s)

        gt_img = Image.open(rgb_dict[idx]).resize((W_s, H_s), Image.LANCZOS)
        gt_tensor = tf.to_tensor(gt_img).cuda()

        cam = GSCamera(
            colmap_id=ti, R=R.T, T=T,
            FoVx=FovX, FoVy=FovY,
            image=gt_tensor, gt_alpha_mask=None,
            image_name=f"test_{idx}", image_path=rgb_dict[idx], uid=ti,
            data_device="cuda"
        )

        with torch.no_grad():
            out = gs_render(cam, gaussians, pipe, bg_color, kernel_size=kernel_size,
                           require_depth=True)
            rendered_depth = out["expected_depth"].squeeze(0).cpu().numpy()

        depth_src = depth_dict.get(idx)
        if depth_src is None:
            continue
        gt_depth = cv2.imread(depth_src, cv2.IMREAD_UNCHANGED).astype(np.float32) * DEPTH_SCALE
        gt_depth = cv2.resize(gt_depth, (W_s, H_s), interpolation=cv2.INTER_NEAREST)

        valid = gt_depth > 0
        if valid.sum() < 100:
            continue

        pred = rendered_depth[valid] * global_scale
        gt = gt_depth[valid]

        per_view['idx'].append(idx)
        per_view['abs_rel'].append(np.mean(np.abs(pred - gt) / gt))
        per_view['sq_rel'].append(np.mean(((pred - gt) ** 2) / gt))
        per_view['rmse'].append(np.sqrt(np.mean((pred - gt) ** 2)))

        ratio = np.maximum(pred / gt, gt / pred)
        per_view['delta_1'].append(np.mean(ratio < 1.25))
        per_view['delta_2'].append(np.mean(ratio < 1.25 ** 2))
        per_view['delta_3'].append(np.mean(ratio < 1.25 ** 3))

        per_view['scale_factor'].append(np.median(rendered_depth[valid]) / np.median(gt_depth[valid]))

        pred_full = rendered_depth * global_scale
        pred_normals = depth_to_normals(pred_full, fx_s, fy_s)
        gt_normals = depth_to_normals(gt_depth, fx_s, fy_s)
        cos_sim = np.clip(np.sum(pred_normals * gt_normals, axis=-1)[valid], -1, 1)
        per_view['normal_acc'].append(np.mean(np.degrees(np.arccos(cos_sim)) < 30))

        n_sample = min(5000, len(gt))
        sidx = np.random.choice(len(gt), n_sample, replace=False)
        all_gt_depths_sampled.append(gt[sidx])
        all_errors_sampled.append(np.abs(pred[sidx] - gt[sidx]))
        all_rel_errors_sampled.append(np.abs(pred[sidx] - gt[sidx]) / gt[sidx])

    for k in per_view:
        per_view[k] = np.array(per_view[k])
    all_gt_depths_flat = np.concatenate(all_gt_depths_sampled)
    all_errors_flat = np.concatenate(all_errors_sampled)
    all_rel_errors_flat = np.concatenate(all_rel_errors_sampled)

    print(f"\n  Global scale:  {global_scale:.4f}")
    print(f"  AbsRel:        {per_view['abs_rel'].mean():.4f}  (std: {per_view['abs_rel'].std():.4f})")
    print(f"  SqRel:         {per_view['sq_rel'].mean():.4f}")
    print(f"  RMSE:          {per_view['rmse'].mean():.4f}  (std: {per_view['rmse'].std():.4f})")
    print(f"  delta<1.25:    {per_view['delta_1'].mean()*100:.2f}%")
    print(f"  delta<1.25^2:  {per_view['delta_2'].mean()*100:.2f}%")
    print(f"  delta<1.25^3:  {per_view['delta_3'].mean()*100:.2f}%")
    print(f"  Normal<30:     {per_view['normal_acc'].mean()*100:.2f}%")

    geo_results = {
        "iteration": eval_iter,
        "num_gaussians": gaussians.get_xyz.shape[0],
        "global_scale": float(global_scale),
        "AbsRel": float(per_view['abs_rel'].mean()),
        "SqRel": float(per_view['sq_rel'].mean()),
        "RMSE": float(per_view['rmse'].mean()),
        "delta_1.25": float(per_view['delta_1'].mean()),
        "delta_1.25^2": float(per_view['delta_2'].mean()),
        "delta_1.25^3": float(per_view['delta_3'].mean()),
        "normal_acc_30": float(per_view['normal_acc'].mean()),
    }
    with open(os.path.join(MODEL_PATH, f"geo_results_iter{eval_iter}.json"), "w") as f:
        json.dump(geo_results, f, indent=2)
    print(f"  저장: geo_results_iter{eval_iter}.json")

In [ ]:
# Per-view 분포 분석
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# (a) AbsRel per view
ax = axes[0, 0]
ax.bar(range(len(per_view['abs_rel'])), per_view['abs_rel'], color='steelblue', alpha=0.7, width=0.8)
ax.axhline(per_view['abs_rel'].mean(), color='red', linestyle='--', linewidth=1.5,
           label=f"mean={per_view['abs_rel'].mean():.4f}")
ax.set_xlabel('Test view index')
ax.set_ylabel('AbsRel')
ax.set_title('(a) AbsRel per view')
ax.legend()

# (b) RMSE per view
ax = axes[0, 1]
ax.bar(range(len(per_view['rmse'])), per_view['rmse'], color='coral', alpha=0.7, width=0.8)
ax.axhline(per_view['rmse'].mean(), color='red', linestyle='--', linewidth=1.5,
           label=f"mean={per_view['rmse'].mean():.2f}")
ax.set_xlabel('Test view index')
ax.set_ylabel('RMSE')
ax.set_title('(b) RMSE per view')
ax.legend()

# (c) δ thresholds per view
ax = axes[0, 2]
x = range(len(per_view['delta_1']))
ax.plot(x, per_view['delta_1'] * 100, 'o-', markersize=3, label='δ<1.25')
ax.plot(x, per_view['delta_2'] * 100, 's-', markersize=3, label='δ<1.25²')
ax.plot(x, per_view['delta_3'] * 100, '^-', markersize=3, label='δ<1.25³')
ax.set_xlabel('Test view index')
ax.set_ylabel('Accuracy (%)')
ax.set_title('(c) δ accuracy per view')
ax.legend()
ax.set_ylim(0, 105)

# (d) Per-view scale factor
ax = axes[1, 0]
ax.scatter(range(len(per_view['scale_factor'])), per_view['scale_factor'], c='green', s=20, zorder=3)
ax.axhline(1.0, color='black', linestyle='-', alpha=0.3, label='ideal (1.0)')
sf_mean = per_view['scale_factor'].mean()
sf_std = per_view['scale_factor'].std()
ax.axhline(sf_mean, color='red', linestyle='--', label=f'mean={sf_mean:.3f}')
ax.fill_between(range(len(per_view['scale_factor'])),
                sf_mean - sf_std, sf_mean + sf_std, alpha=0.15, color='red', label=f'±1σ ({sf_std:.4f})')
ax.set_xlabel('Test view index')
ax.set_ylabel('pred_median / gt_median')
ax.set_title('(d) Per-view scale factor')
ax.legend(fontsize=8)

# (e) Normal accuracy per view
ax = axes[1, 1]
ax.bar(range(len(per_view['normal_acc'])), per_view['normal_acc'] * 100,
       color='mediumpurple', alpha=0.7, width=0.8)
ax.axhline(per_view['normal_acc'].mean() * 100, color='red', linestyle='--', linewidth=1.5,
           label=f"mean={per_view['normal_acc'].mean()*100:.1f}%")
ax.set_xlabel('Test view index')
ax.set_ylabel('% normals within 30°')
ax.set_title('(e) Normal accuracy per view')
ax.legend()

# (f) Metric box plots
ax = axes[1, 2]
data = [per_view['abs_rel'], per_view['rmse'] / max(per_view['rmse'].max(), 1e-8),
        per_view['delta_1'], per_view['normal_acc']]
bp = ax.boxplot(data, labels=['AbsRel', 'RMSE\n(norm)', 'δ<1.25', 'Normal\n<30°'],
                patch_artist=True, widths=0.6)
colors = ['steelblue', 'coral', 'seagreen', 'mediumpurple']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)
ax.set_title('(f) Metric distributions')
ax.set_ylabel('Value')

plt.suptitle('Per-View 기하학 분석', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Error vs Depth 거리 + 누적 오차 분석
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) 거리별 상대 오차 (binned)
ax = axes[0]
n_bins = 20
gt_min, gt_max = np.percentile(all_gt_depths_flat, [1, 99])
bin_edges = np.linspace(gt_min, gt_max, n_bins + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_means, bin_stds = [], []
for i in range(n_bins):
    mask = (all_gt_depths_flat >= bin_edges[i]) & (all_gt_depths_flat < bin_edges[i+1])
    if mask.sum() > 10:
        bin_means.append(all_rel_errors_flat[mask].mean())
        bin_stds.append(all_rel_errors_flat[mask].std())
    else:
        bin_means.append(np.nan)
        bin_stds.append(np.nan)
bin_means, bin_stds = np.array(bin_means), np.array(bin_stds)

ax.plot(bin_centers, bin_means, 'o-', color='steelblue', markersize=4)
valid_bins = ~np.isnan(bin_means)
ax.fill_between(bin_centers[valid_bins],
                (bin_means - bin_stds)[valid_bins], (bin_means + bin_stds)[valid_bins],
                alpha=0.2, color='steelblue')
ax.set_xlabel('GT Depth')
ax.set_ylabel('Relative Error (|pred-gt|/gt)')
ax.set_title('(a) 거리별 상대 오차')
ax.grid(True, alpha=0.3)

# (b) 누적 오차 분포
ax = axes[1]
sorted_rel_errors = np.sort(all_rel_errors_flat)
cumulative = np.arange(1, len(sorted_rel_errors) + 1) / len(sorted_rel_errors) * 100
step = max(1, len(sorted_rel_errors) // 5000)
ax.plot(sorted_rel_errors[::step] * 100, cumulative[::step], color='steelblue', linewidth=2)

for th, color in [(5, 'green'), (10, 'orange'), (25, 'red')]:
    pct = (all_rel_errors_flat < th / 100).mean() * 100
    ax.axvline(th, color=color, linestyle='--', alpha=0.5)
    ax.text(th + 0.5, pct - 5, f'{pct:.1f}%\n@{th}%', fontsize=9, color=color)

ax.set_xlabel('Relative Error (%)')
ax.set_ylabel('Cumulative % of pixels')
ax.set_title('(b) 누적 오차 분포')
ax.set_xlim(0, 50)
ax.grid(True, alpha=0.3)

# (c) 절대 오차 히스토그램
ax = axes[2]
clip_val = np.percentile(all_errors_flat, 95)
ax.hist(all_errors_flat[all_errors_flat < clip_val], bins=50, color='coral', alpha=0.7, density=True)
ax.axvline(np.median(all_errors_flat), color='blue', linestyle='--',
           label=f'median={np.median(all_errors_flat):.2f}')
ax.axvline(np.mean(all_errors_flat), color='red', linestyle='--',
           label=f'mean={np.mean(all_errors_flat):.2f}')
ax.set_xlabel('Absolute Error')
ax.set_ylabel('Density')
ax.set_title('(c) 절대 오차 분포 (95th pctl clip)')
ax.legend()

plt.suptitle('Depth Error 상세 분석', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"픽셀 기준 통계 ({len(all_rel_errors_flat):,} samples):")
print(f"  5% 이내 오차:  {(all_rel_errors_flat < 0.05).mean()*100:.1f}%")
print(f"  10% 이내 오차: {(all_rel_errors_flat < 0.10).mean()*100:.1f}%")
print(f"  25% 이내 오차: {(all_rel_errors_flat < 0.25).mean()*100:.1f}%")
print(f"  Median abs error: {np.median(all_errors_flat):.4f}")
print(f"  Mean abs error:   {np.mean(all_errors_flat):.4f}")

In [ ]:
# Depth 비교 시각화: 렌더링 vs GT
import matplotlib.pyplot as plt

# 첫 4개 test 뷰의 depth 비교
n_vis = min(4, len(test_indices))
fig, axes = plt.subplots(n_vis, 3, figsize=(15, 5*n_vis))
if n_vis == 1: axes = axes[np.newaxis, :]

axes[0, 0].set_title('Rendered Depth', fontsize=14, fontweight='bold')
axes[0, 1].set_title('GT Depth', fontsize=14, fontweight='bold')
axes[0, 2].set_title('|Error|', fontsize=14, fontweight='bold')

for vi in range(n_vis):
    idx = test_indices[vi]
    K, R, T, W, H = parse_calib(calib_dict[idx])
    scale = RESOLUTION
    W_s, H_s = W // scale, H // scale
    fx_s, fy_s = K[0,0] / scale, K[1,1] / scale

    gt_img = Image.open(rgb_dict[idx]).resize((W_s, H_s), Image.LANCZOS)
    gt_tensor = tf.to_tensor(gt_img).cuda()

    cam = GSCamera(
        colmap_id=vi, R=R.T, T=T,
        FoVx=focal2fov(fx_s, W_s), FoVy=focal2fov(fy_s, H_s),
        image=gt_tensor, gt_alpha_mask=None,
        image_name=f"vis_{idx}", image_path=rgb_dict[idx], uid=vi,
        data_device="cuda"
    )

    with torch.no_grad():
        out = gs_render(cam, gaussians, pipe, bg_color, kernel_size=kernel_size,
                       require_depth=True)
        rd = out["expected_depth"].squeeze(0).cpu().numpy()

    gt_d = cv2.imread(depth_dict[idx], cv2.IMREAD_UNCHANGED).astype(np.float32) * DEPTH_SCALE
    gt_d = cv2.resize(gt_d, (W_s, H_s), interpolation=cv2.INTER_NEAREST)

    valid = gt_d > 0
    vmin, vmax = gt_d[valid].min(), gt_d[valid].max()

    axes[vi, 0].imshow(rd * global_scale, cmap='viridis', vmin=vmin, vmax=vmax)
    axes[vi, 0].set_ylabel(f'test_{idx}', fontsize=10)
    axes[vi, 0].axis('off')

    axes[vi, 1].imshow(gt_d, cmap='viridis', vmin=vmin, vmax=vmax)
    axes[vi, 1].axis('off')

    # Scale alignment
    rd_aligned = rd * global_scale
    error = np.abs(rd_aligned - gt_d) * valid
    axes[vi, 2].imshow(error, cmap='hot', vmin=0, vmax=np.percentile(error[valid], 95))
    axes[vi, 2].axis('off')

plt.suptitle(f'Depth 비교 (AbsRel: {per_view["abs_rel"].mean():.4f})', fontsize=16)
plt.tight_layout()
plt.show()

### Pseudo GT 메쉬 vs 추출 메쉬 (Chamfer Distance)

In [ ]:
# Pseudo GT 메쉬 생성 (GT depth back-projection) → Chamfer Distance
from scipy.spatial import cKDTree

# 1. Pseudo GT 포인트 클라우드 (test 뷰의 GT depth 사용)
pseudo_gt_points = []
for idx in tqdm(test_indices + train_indices, desc="Pseudo GT 생성"):
    depth_src = depth_dict.get(idx)
    if depth_src is None:
        continue
    K, R, T, W, H = parse_calib(calib_dict[idx])
    gt_d = cv2.imread(depth_src, cv2.IMREAD_UNCHANGED).astype(np.float32) * DEPTH_SCALE

    valid = gt_d > 0
    ys, xs = np.where(valid)

    # 서브샘플링
    if len(xs) > 1000:
        choice = np.random.choice(len(xs), 1000, replace=False)
        xs, ys = xs[choice], ys[choice]

    depths = gt_d[ys, xs]
    pts_cam = np.stack([
        (xs - K[0,2]) / K[0,0] * depths,
        (ys - K[1,2]) / K[1,1] * depths,
        depths
    ], axis=1)
    pts_world = (R.T @ (pts_cam.T - T[:, None])).T
    pseudo_gt_points.append(pts_world)

pseudo_gt = np.concatenate(pseudo_gt_points, axis=0).astype(np.float32)

# 중복 제거 (voxel downsampling)
import open3d as o3d
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(pseudo_gt)
pcd = pcd.voxel_down_sample(voxel_size=0.5)
pseudo_gt = np.asarray(pcd.points).astype(np.float32)
print(f"Pseudo GT 포인트: {len(pseudo_gt):,}개")

# 2. 추출된 메쉬에서 포인트 샘플링
mesh_files = sorted(glob.glob(f"{MODEL_PATH}/**/fuse_*_post.ply", recursive=True))
if not mesh_files:
    mesh_files = sorted(glob.glob(f"{MODEL_PATH}/**/fuse_*.ply", recursive=True))
if not mesh_files:
    print("메쉬 파일을 찾을 수 없습니다. Section 3을 먼저 실행하세요.")
else:
    mesh = trimesh.load(mesh_files[0])
    pred_points = mesh.sample(min(200000, len(mesh.faces)))

    # 오브젝트 영역으로 크롭 (pseudo GT bounding box 기준)
    margin = 10.0
    bbox_min = pseudo_gt.min(axis=0) - margin
    bbox_max = pseudo_gt.max(axis=0) + margin
    mask = np.all((pred_points >= bbox_min) & (pred_points <= bbox_max), axis=1)
    pred_points = pred_points[mask]
    print(f"크롭 후 메쉬 포인트: {len(pred_points):,}개")
    print(f"추출 메쉬 샘플 포인트: {len(pred_points):,}개")

    # 3. Chamfer Distance 계산
    # pred → gt
    tree_gt = cKDTree(pseudo_gt)
    d_pred_to_gt, _ = tree_gt.query(pred_points)

    # gt → pred
    tree_pred = cKDTree(pred_points)
    d_gt_to_pred, _ = tree_pred.query(pseudo_gt)

    chamfer = (d_pred_to_gt.mean() + d_gt_to_pred.mean()) / 2
    accuracy = d_pred_to_gt.mean()    # pred가 gt에 얼마나 가까운지
    completeness = d_gt_to_pred.mean() # gt가 pred에 얼마나 커버되는지

    print(f"\n{'='*60}")
    print(f"Chamfer Distance (Pseudo GT 기준)")
    print(f"{'='*60}")
    print(f"  Chamfer Distance: {chamfer:.4f}")
    print(f"  Accuracy:         {accuracy:.4f}  (pred→gt)")
    print(f"  Completeness:     {completeness:.4f}  (gt→pred)")

    # 결과 저장
    test_results["chamfer_metrics"] = {
        "chamfer_distance": float(chamfer),
        "accuracy": float(accuracy),
        "completeness": float(completeness),
    }
    with open(os.path.join(MODEL_PATH, "test_results.json"), 'w') as f:
        json.dump(test_results, f, indent=2)
    print(f"저장: {MODEL_PATH}/test_results.json")

---
## 5. 시각화

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

render_files = sorted(glob.glob(f"{test_render_dir}/*.png"))[:6]
if render_files:
    n = len(render_files)
    fig, axes = plt.subplots(n, 2, figsize=(12, 5*n))
    if n == 1: axes = axes[np.newaxis, :]
    axes[0, 0].set_title("Rendered", fontsize=14, fontweight='bold')
    axes[0, 1].set_title("Ground Truth", fontsize=14, fontweight='bold')
    for i, rf in enumerate(render_files):
        fname = os.path.basename(rf)
        axes[i, 0].imshow(Image.open(rf)); axes[i, 0].set_ylabel(fname); axes[i, 0].axis('off')
        gt = os.path.join(test_gt_dir, fname)
        if os.path.exists(gt): axes[i, 1].imshow(Image.open(gt))
        axes[i, 1].axis('off')
    plt.suptitle(f"Test Views (PSNR: {np.mean(psnrs):.2f} dB)", fontsize=16)
    plt.tight_layout()
    plt.show()

---
## 6. 결과 저장

In [ ]:
import json, zipfile

experiment_config = {
    "dataset": {
        "total_images": N_ALL,
        "num_train": len(train_indices),
        "num_test": len(test_indices),
        "resolution": RESOLUTION,
        "depth_supervision": True,
        "depth_scale": DEPTH_SCALE,
        "init_points": "depth_backprojection",
    },
    "training": {
        "iterations": ITERATIONS,
        "lambda_gt_depth": LAMBDA_GT_DEPTH,
        "tgpc_weight": TGPC_WEIGHT,
        "lambda_dssim": LAMBDA_DSSIM,
        "lambda_depth_normal": LAMBDA_DEPTH_NORMAL,
        "lambda_geo": LAMBDA_GEO,
        "lambda_ncc": LAMBDA_NCC,
    },
    "test_metrics": test_results["test_metrics"],
}

with open(os.path.join(MODEL_PATH, "experiment_config.json"), 'w') as f:
    json.dump(experiment_config, f, indent=2)

ZIP_RESULT = f"{OUTPUT_DIR}/triags_depth_results.zip"
with zipfile.ZipFile(ZIP_RESULT, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(MODEL_PATH):
        if 'point_cloud' in root and not any(f'iteration_{it}' in root for it in [25000, ITERATIONS]):
            continue
        for file in files:
            if file.endswith('.pth'):
                continue
            full_path = os.path.join(root, file)
            zf.write(full_path, os.path.relpath(full_path, OUTPUT_DIR))

print(f"zip: {ZIP_RESULT} ({os.path.getsize(ZIP_RESULT)/1e6:.1f} MB)")

try:
    from google.colab import files
    files.download(ZIP_RESULT)
except ImportError:
    print(f"로컬: {ZIP_RESULT}")

In [ ]:
SAVE_TO_DRIVE = False  #@param {type:"boolean"}

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = "/content/drive/MyDrive/triags_results"
    os.makedirs(DRIVE_DIR, exist_ok=True)
    shutil.copy2(ZIP_RESULT, DRIVE_DIR)
    print(f"저장 완료: {DRIVE_DIR}")
else:
    print("Drive 저장 건너뜀")